# US Valuation Pipeline (Flat Script)

- Generated: 2025-11-04T05:52:18.065698Z
- 목적: 일반 스크립트 형태로 납작하게(flat) 구성된 파이프라인.
- 사용법: 아래 코드셀만 실행하면 됩니다. 필요 파라미터는 코드 상단에서 수정하세요.

In [53]:
# -*- coding: utf-8 -*-

"""
us_valuation_pipeline_flat_v2.py (emoji-free)
- 목적: 단일/복수 티커 for-loop로 일괄 실행하는 '납작한(flat)' 스크립트
- 반영: revenue_forecast_step, psr_forecast_step 파라미터 / revenue_forecast_df.ffill(limit=2)
- 외부 모듈: DATA.us_* (sarima, lstm, prophet, es), DATA.stock_invest_function 등
"""

import sys, os, gc, time, warnings, traceback, datetime as dt
from pathlib import Path
import importlib
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from pandas.tseries.offsets import MonthEnd
import requests

warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────────────────────────────────────
# 0) 경로 설정: repo 루트 찾아서 sys.path에 추가
# ──────────────────────────────────────────────────────────────────────────────
def add_repo_path():
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "DATA").exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            print(f"[INFO] Project root added to sys.path: {p}")
            return str(p)
    fallback = r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast"
    if os.path.isdir(fallback) and fallback not in sys.path:
        sys.path.insert(0, fallback)
        print(f"[WARNING] Using fallback path: {fallback}")
        return fallback
    raise FileNotFoundError("DATA 폴더를 찾을 수 없습니다.")

PROJECT_ROOT = add_repo_path()

# ──────────────────────────────────────────────────────────────────────────────
# 1) 외부 모듈 import (유지)
# ──────────────────────────────────────────────────────────────────────────────
import DATA.us_sarima_forecast as sarima
importlib.reload(sarima)

import DATA.us_lstm_forecast_v2 as lstm_v2
importlib.reload(lstm_v2)

import DATA.us_prophet_forecast_v3 as prophet_v3
importlib.reload(prophet_v3)

import DATA.us_est_forecast_v2 as esmod
importlib.reload(esmod)

from DATA.stock_invest_function import *
# from DATA.us_target_ticker_list import ticker_list

# ──────────────────────────────────────────────────────────────────────────────
# 2) 최소 유틸
# ──────────────────────────────────────────────────────────────────────────────
def log(stage: str, msg: str):
    print(f"[{dt.datetime.now().strftime('%H:%M:%S')}] {stage}: {msg}")

def to_month_end_safe(s: pd.Series) -> pd.Series:
    s = pd.to_datetime(s, errors="coerce")
    prev_mask = s.dt.day.between(1, 5, inclusive="both")
    out = s.copy()
    out.loc[prev_mask] = (s.loc[prev_mask] + MonthEnd(-1))
    out.loc[~prev_mask] = (s.loc[~prev_mask] + MonthEnd(0))
    return out

# ──────────────────────────────────────────────────────────────────────────────
# 3) 사용자 파라미터
# ──────────────────────────────────────────────────────────────────────────────
API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TARGET_TICKERS = ['AAPL']
BATCH_SIZE = 20
revenue_forecast_step = 6
psr_forecast_step = 18

# ──────────────────────────────────────────────────────────────────────────────
# 4) DB 엔진/DDL
# ──────────────────────────────────────────────────────────────────────────────
eng = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
)

ddl_val_forecast = """
CREATE TABLE IF NOT EXISTS `us_valuation_forecast_result` (
  `id` BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  `date` DATE NOT NULL,
  `ticker` VARCHAR(16) NOT NULL,
  `indicator` VARCHAR(128) NOT NULL,
  `value` DECIMAL(20,8) NULL,
  `forecate_date` DATE NOT NULL,
  PRIMARY KEY (`id`),
  UNIQUE KEY `uq_tk_fc_ind` (`ticker`, `forecate_date`, `indicator`),
  KEY `idx_tk_date` (`ticker`, `date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""


ddl_rev = """
CREATE TABLE IF NOT EXISTS `us_revenue_forecast_result` (
  `id` BIGINT UNSIGNED NOT NULL AUTO_INCREMENT,
  `ticker` VARCHAR(16) NOT NULL,
  `date_month_end` DATE NOT NULL,
  `revenue_billions_sarima_noexog` DECIMAL(20,8) NULL,
  `revenue_billions_lstm_forecast` DECIMAL(20,8) NULL,
  `revenue_billions_prophet_forecast` DECIMAL(20,8) NULL,
  `revenue_billions_esq_forecast` DECIMAL(20,8) NULL,
  `created_at` DATETIME NOT NULL,
  `created_ts` TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
  `updated_ts` TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  KEY `idx_ticker_date` (`ticker`, `date_month_end`),
  KEY `idx_created_at` (`created_at`),
  UNIQUE KEY `uq_ticker_date` (`ticker`, `date_month_end`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
"""

with eng.begin() as conn:
    conn.execute(text(ddl_rev))
    conn.execute(text(ddl_val_forecast))   # ← 추가
    log("DDL", "Tables ensured: us_valuation_result, us_revenue_forecast_result, us_valuation_forecast_result")

# ──────────────────────────────────────────────────────────────────────────────
# 5) 루프 상태 변수
# ──────────────────────────────────────────────────────────────────────────────
batch_results = []
batch_revenue_results = []
total_upsert_rows = 0
total_revenue_rows = 0
total_success_tickers = 0
error_ticker_list = []

# ──────────────────────────────────────────────────────────────────────────────
# 6) for-loop: 티커별 수행
# ──────────────────────────────────────────────────────────────────────────────
TARGET_TICKERS = ['INVX', 'AAPL', 'MU']

for idx, ticker in enumerate(TARGET_TICKERS, 1):
    log("TICKER", f"{idx}/{len(TARGET_TICKERS)} {ticker}")
    try:
        # (A) FMP 분기 매출 수집
        url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
        params = {'limit': 200, 'apikey': API_KEY, 'period': 'quarter'}
        r = requests.get(url, params=params, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"FMP revenue HTTP {r.status_code}")
        data = r.json()
        if not data:
            raise RuntimeError("FMP revenue: empty")

        fmp_revenue_df = pd.DataFrame([{
            'ticker': ticker,
            'date': it.get('date', ''),
            'calendar_year': it.get('calendarYear', ''),
            'period': it.get('period', ''),
            'revenue': it.get('revenue', 0) or 0,
            'revenue_billions': round((it.get('revenue', 0) or 0) / 1_000_000_000, 2),
        } for it in data])
        fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
        fmp_revenue_df['date_month_end'] = to_month_end_safe(fmp_revenue_df['date'])
        fmp_revenue_df = fmp_revenue_df.dropna(subset=['date_month_end']).drop_duplicates(subset=['date_month_end'])

        # (B) DB 분기 매출 조인
        clean_ticker = ticker.strip().upper()
        sql_q = text("""
            SELECT date, ticker, saleq
              FROM US_fundq
             WHERE UPPER(ticker)=:t
               AND saleq IS NOT NULL
          ORDER BY date ASC
        """)
        with eng.connect() as conn:
            db_revenue_raw = pd.read_sql(sql_q, conn, params={"t": clean_ticker})
        if not db_revenue_raw.empty:
            db_revenue_raw['date'] = pd.to_datetime(db_revenue_raw['date'], errors='coerce')
            db_revenue_raw = db_revenue_raw.dropna(subset=['date'])
            db_revenue_raw['revenue_billions'] = db_revenue_raw['saleq'] / 1000.0
            db_revenue_raw['date_month_end'] = to_month_end_safe(db_revenue_raw['date'])
            db_revenue_df = db_revenue_raw.loc[
                db_revenue_raw['revenue_billions'] != db_revenue_raw['revenue_billions'].shift()
            ][['ticker', 'date_month_end', 'revenue_billions']]
        else:
            db_revenue_df = pd.DataFrame(columns=['ticker', 'date_month_end', 'revenue_billions'])

        rev_data = pd.merge(
            fmp_revenue_df[['ticker','date_month_end','revenue_billions','calendar_year','period']],
            db_revenue_df, on=['ticker','date_month_end'], how='outer', suffixes=('_fmp','_db')
        )
        if 'revenue_billions_fmp' in rev_data.columns:
            rev_data['revenue_billions'] = rev_data['revenue_billions_fmp'].fillna(rev_data['revenue_billions_db'])
        rev_data = rev_data.drop(columns=[c for c in ['revenue_billions_fmp','revenue_billions_db'] if c in rev_data.columns])
        rev_data = rev_data.dropna(subset=['date_month_end']).sort_values('date_month_end')

        # (C) 매출 예측
        sarima_df, _ = sarima.run_sarima_prediction(rev_data, forecast_quarters=revenue_forecast_step, exog_col=None)
        sarima_df = sarima_df.sort_values("date_month_end").set_index("date_month_end")
        lstm_raw_df, _ = lstm_v2.run_lstm_revenue_prediction(rev_data, ticker=ticker, prediction_quarters=revenue_forecast_step)
        prophet_raw_df, _ = prophet_v3.run_prophet_revenue_only(rev_data, ticker=ticker, prediction_quarters=revenue_forecast_step)
        es_raw_df, _ = esmod.run_es_revenue_quarterly(rev_data, ticker=ticker, prediction_quarters=revenue_forecast_step)

        # (D) FMP 월말 시총 수집
        all_mc = []
        for year in range(2010, dt.datetime.now().year + 1):
            url_mc = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
            params_mc = {'from': f"{year}-01-01", 'to': f"{year}-12-31", 'apikey': API_KEY}
            rr = requests.get(url_mc, params=params_mc, timeout=30)
            if rr.status_code == 200:
                j = rr.json()
                if isinstance(j, list) and j:
                    all_mc.extend(j)
            time.sleep(0.3)

        df_mc = pd.DataFrame(all_mc)
        df_mc['date'] = pd.to_datetime(df_mc['date'])
        df_mc['date_month_end'] = to_month_end_safe(df_mc['date'])
        df_mc = df_mc.drop_duplicates(subset=['date_month_end']).sort_values('date_month_end')
        df_mc['market_cap_billions'] = (df_mc['marketCap'] / 1_000_000_000).round(2)

        # (E) PSR 계산
        enhanced = pd.merge(
            df_mc[['date_month_end','market_cap_billions']],
            rev_data[['date_month_end','ticker','revenue_billions']],
            on='date_month_end', how='outer'
        ).sort_values('date_month_end')
        enhanced['ticker'] = enhanced['ticker'].ffill().bfill()
        enhanced['revenue_ttm'] = (
            enhanced.groupby('ticker')['revenue_billions']
                    .rolling(window=4, min_periods=1)
                    .sum()
                    .reset_index(level=0, drop=True)
        )
        enhanced['revenue_ttm_shift'] = enhanced.groupby('ticker')['revenue_ttm'].shift(2)
        enhanced['PSR_ttm'] = enhanced['market_cap_billions'] / enhanced['revenue_ttm_shift']
        enhanced = enhanced.replace([np.inf, -np.inf], np.nan).dropna(subset=['PSR_ttm'])

        psr_series_ok = enhanced[['date_month_end','PSR_ttm']].dropna()
        if psr_series_ok.empty or psr_series_ok['PSR_ttm'].count() < 6:
            raise RuntimeError("PSR series too short after TTM shift")

        # (F) PSR 예측
        psr_sarima_df, _ = sarima.run_sarima_psr_only(
            enhanced, periods=psr_forecast_step, target_col="PSR_ttm",
            analysis_start="2012-06-01", warmup_months=6, fill_method="interpolate", ic="aic"
        )
        psr_lstm_df, _   = lstm_v2.run_lstm_psr_prediction(enhanced, ticker=ticker, prediction_months=psr_forecast_step)
        psr_prophet_df, _= prophet_v3.run_prophet_psr_only(enhanced, ticker=ticker, prediction_months=psr_forecast_step)
        psr_es_df, _     = esmod.run_es_psr_only(enhanced, ticker=ticker, prediction_months=psr_forecast_step)

        # (G) TTM 매출+PSR 결합 → valuation_df
        def _pick(df, cols):
            d = df.copy()
            if 'date_month_end' not in d.columns:
                d = d.reset_index()
                if 'date_month_end' not in d.columns and 'index' in d.columns:
                    d = d.rename(columns={'index':'date_month_end'})
            d = d.drop_duplicates(subset=['date_month_end']).sort_values('date_month_end')
            return d[['date_month_end'] + cols].set_index('date_month_end')

        rev_sarima = _pick(sarima_df,     ['revenue_billions_sarima_noexog'])
        rev_lstm   = _pick(lstm_raw_df,   ['revenue_billions_lstm_forecast'])
        rev_prop   = _pick(prophet_raw_df,['revenue_billions_prophet_forecast'])
        rev_es     = _pick(es_raw_df,     ['revenue_billions_esq_forecast'])

        revenue_forecast_df = pd.concat([rev_sarima, rev_lstm, rev_prop, rev_es], axis=1, join='outer').reset_index()
        revenue_forecast_df['ticker'] = ticker
        revenue_forecast_df = revenue_forecast_df.ffill(limit=2)
        # revenue_forecast_df = revenue_forecast_df.rename(columns={'date_month_end': 'date'})

        rf_for_db = revenue_forecast_df[['date_month_end','revenue_billions_sarima_noexog',
                                         'revenue_billions_lstm_forecast',
                                         'revenue_billions_prophet_forecast',
                                         'revenue_billions_esq_forecast']].copy()
        rf_for_db['ticker'] = ticker
        batch_revenue_results.append(rf_for_db)

        tmp = revenue_forecast_df.copy()
        tmp['date_month_end'] = pd.to_datetime(tmp['date_month_end'])
        tmp = tmp.sort_values('date_month_end')
        rev_cols_level = [c for c in tmp.columns if 'revenue_billions' in c and not c.endswith('_ttm')]
        for c in rev_cols_level:
            tmp[f"{c}_ttm"] = (
                tmp.groupby('ticker', group_keys=False)[c]
                   .rolling(window=4, min_periods=1)
                   .sum()
                   .reset_index(level=0, drop=True)
            )
        revenue_cols_ttm = [
            'revenue_billions_sarima_noexog_ttm',
            'revenue_billions_lstm_forecast_ttm',
            'revenue_billions_prophet_forecast_ttm',
            'revenue_billions_esq_forecast_ttm'
        ]
        tmp['revenue_billions_avg_of_4_ttm'] = tmp[revenue_cols_ttm].mean(axis=1)

        psr_sarima = _pick(psr_sarima_df, ['PSR_ttm_sarima_forecast'])
        psr_lstm   = _pick(psr_lstm_df,   ['PSR_ttm_lstm_forecast'])
        psr_prop   = _pick(psr_prophet_df,['PSR_prophet_forecast_noexog'])
        psr_es     = _pick(psr_es_df,     ['PSR_es_forecast'])
        psr_forecast_df = pd.concat([psr_sarima, psr_lstm, psr_prop, psr_es], axis=1, join='outer').reset_index()

        valuation_df = pd.merge(
            tmp.filter(regex="date_month_end|ticker|_ttm$"),
            psr_forecast_df, on='date_month_end', how='inner'
        ).sort_values('date_month_end')

        cols_to_ffill = ['ticker'] + [c for c in valuation_df.columns if 'revenue_billions' in c]
        valuation_df[cols_to_ffill] = valuation_df[cols_to_ffill].ffill(limit=2)

        valuation_df['sarima_valuation']  = valuation_df['revenue_billions_sarima_noexog_ttm']  * valuation_df['PSR_ttm_sarima_forecast']
        valuation_df['lstm_valuation']    = valuation_df['revenue_billions_lstm_forecast_ttm']  * valuation_df['PSR_ttm_lstm_forecast']
        valuation_df['prophet_valuation'] = valuation_df['revenue_billions_prophet_forecast_ttm'] * valuation_df['PSR_prophet_forecast_noexog']
        valuation_df['es_valuation']      = valuation_df['revenue_billions_esq_forecast_ttm']   * valuation_df['PSR_es_forecast']

        valuation_pack = (
            valuation_df
            .groupby('ticker', group_keys=False)
            .apply(lambda d: d.tail(30))
            .reset_index(drop=True)
        )
        batch_results.append(valuation_pack)
        total_success_tickers += 1
        log("OK-VAL-PACK", f"{ticker} packed={len(valuation_pack)} batch={len(batch_results)}")

    except Exception as e:
        log("EXC", f"{ticker} stage failed: {e}")
        error_ticker_list.append({'ticker': ticker, 'error': str(e), 'tb': traceback.format_exc().splitlines()[-1]})
        continue

    try:
        is_last = (idx == len(TARGET_TICKERS))
        if (len(batch_results) >= BATCH_SIZE) or is_last:
            log("BATCH-FLUSH", f"valuation={len(batch_results)}, revenue={len(batch_revenue_results)}, is_last={is_last}")

                # ── [A] valuation_df들을 long-format으로 변환하여 us_valuation_forecast_result에 저장 ──
            if batch_results:
                val_concat = pd.concat(batch_results, axis=0, ignore_index=True)

                # date 컬럼 통일
                if 'date' not in val_concat.columns and 'date_month_end' in val_concat.columns:
                    val_concat = val_concat.rename(columns={'date_month_end': 'date'})
                val_concat['date'] = pd.to_datetime(val_concat['date'], errors='coerce')

                # ticker 보장
                if 'ticker' not in val_concat.columns:
                    raise RuntimeError("valuation 저장을 위해 'ticker' 컬럼이 필요합니다.")

                # 저장 대상 컬럼만 선택 (불필요한 컬럼로 인한 NaT/NaN 오염 방지)
                keep_cols = ['date', 'ticker',
                             'sarima_valuation', 'lstm_valuation',
                             'prophet_valuation', 'es_valuation']
                missing = [c for c in keep_cols if c not in val_concat.columns]
                if missing:
                    raise RuntimeError(f"valuation_df에 필요한 컬럼이 없습니다: {missing}")

                vdf = val_concat[keep_cols].copy()
                vdf = vdf.dropna(subset=['date'])                 # 날짜 필수
                vdf['date'] = vdf['date'].dt.date                 # MySQL DATE로

                # wide -> long
                val_long = (
                    vdf.melt(id_vars=['date','ticker'],
                             value_vars=['sarima_valuation','lstm_valuation',
                                         'prophet_valuation','es_valuation'],
                             var_name='indicator', value_name='value')
                       .dropna(subset=['value'])
                )

                # 아무 것도 없으면 스킵(디버깅 로그)
                if val_long.empty:
                    log("VAL-LONG", "변환 결과 0행 (모든 값이 NaN이거나 최근 30행 필터로 비었습니다)")
                else:
                    # 실행일(UTC-naive) → forecate_date
                    forecate_date = pd.Timestamp.utcnow().tz_localize(None).date()
                    val_long['forecate_date'] = forecate_date

                    # 디버깅: 몇 행/샘플 출력
                    log("VAL-LONG", f"to-insert rows={len(val_long)} sample={val_long.head(3).to_dict(orient='records')}")

                    # DB 저장 (덮어쓰기 규칙: 같은 ticker & forecate_date 전량 삭제 후 삽입)
                    try:
                        with eng.begin() as conn:
                            for tk in val_long['ticker'].dropna().unique():
                                conn.execute(
                                    text("""
                                        DELETE FROM us_valuation_forecast_result
                                         WHERE ticker=:tk AND forecate_date=:fc
                                    """),
                                    {'tk': str(tk), 'fc': forecate_date}
                                )

                            rows = val_long[['date','ticker','indicator','value','forecate_date']] \
                                        .to_dict(orient='records')

                            conn.execute(text("""
                                INSERT INTO us_valuation_forecast_result
                                    (`date`, `ticker`, `indicator`, `value`, `forecate_date`)
                                VALUES
                                    (:date, :ticker, :indicator, :value, :forecate_date)
                            """), rows)

                        total_upsert_rows += len(val_long)
                        log("BATCH-UPLOAD", f"us_valuation_forecast_result rows={len(val_long)} total={total_upsert_rows}")

                    except Exception as e:
                        # 에러 원인 바로 표시
                        log("VAL-UPSERT-EXC", f"{type(e).__name__}: {e}")
                        raise

            if batch_revenue_results:

                rev_concat = pd.concat(batch_revenue_results, axis=0, ignore_index=True)

                # 1) 컬럼 정규화: date_month_end -> date
                if 'date' not in rev_concat.columns and 'date_month_end' in rev_concat.columns:
                    rev_concat = rev_concat.rename(columns={'date_month_end': 'date'})
                elif 'date' in rev_concat.columns and 'date_month_end' in rev_concat.columns:
                    rev_concat['date'] = pd.to_datetime(rev_concat['date'], errors='coerce')
                    rev_concat['date'] = rev_concat['date'].fillna(pd.to_datetime(rev_concat['date_month_end'], errors='coerce'))
                    rev_concat = rev_concat.drop(columns=['date_month_end'])

                # 2) 스키마 기반 베이스 컬럼 확보
                base_cols = [
                    'ticker', 'date',
                    'revenue_billions_sarima_noexog',
                    'revenue_billions_lstm_forecast',
                    'revenue_billions_prophet_forecast',
                    'revenue_billions_esq_forecast',
                ]
                for c in base_cols:
                    if c not in rev_concat.columns:
                        rev_concat[c] = np.nan

                # 3) 타입/중복 정리
                rev_concat['date'] = pd.to_datetime(rev_concat['date'], errors='coerce')
                rev_concat = rev_concat.dropna(subset=['ticker','date'])
                rev_concat = rev_concat.drop_duplicates(subset=['ticker','date'], keep='last')

                 # 4) 와이드 -> 롱 (indicator/value)
                long_df = rev_concat.melt(
                    id_vars=['ticker','date'],
                    value_vars=[
                        'revenue_billions_sarima_noexog',
                        'revenue_billions_lstm_forecast',
                        'revenue_billions_prophet_forecast',
                        'revenue_billions_esq_forecast',
                    ],
                    var_name='indicator',
                    value_name='value'
                )
                long_df = long_df.dropna(subset=['value'])

                # forecast_date, created_at을 tz-naive DATE / DATETIME 으로 변환
                forecast_run_date = pd.Timestamp.utcnow().tz_localize(None).date()   # → DATE 형식 (예: 2025-11-04)
                created_at_run = pd.Timestamp.utcnow().tz_localize(None)             # → DATETIME naive

                long_df['forecast_date'] = forecast_run_date
                long_df['created_at'] = created_at_run

                # 5) 업서트: (ticker, date, indicator) 기준으로 삭제 후 삽입
                with eng.begin() as conn:
                    for _, row in long_df.iterrows():
                        conn.execute(text("""
                            DELETE FROM us_revenue_forecast_result
                             WHERE ticker   = :t
                               AND `date`   = :d
                               AND indicator= :ind
                        """), {'t': str(row['ticker']), 'd': row['date'], 'ind': row['indicator']})

                        conn.execute(text("""
                            INSERT INTO us_revenue_forecast_result
                            (ticker, `date`, indicator, value, forecast_date, created_at)
                            VALUES
                            (:t, :d, :ind, :val, :fd, :ca)
                        """), {
                            't': str(row['ticker']),
                            'd': row['date'],
                            'ind': row['indicator'],
                            'val': float(row['value']),
                            'fd': row['forecast_date'],
                            'ca': row['created_at']
                        })
                total_revenue_rows += len(rev_concat)
                log("BATCH-UPLOAD", f"us_revenue_forecast_result rows={len(rev_concat)} total={total_revenue_rows}")

            batch_results.clear()
            batch_revenue_results.clear()
            gc.collect()

    except Exception as e:
        log("EXC-BATCH-FLUSH", f"{ticker} e={e}")
        error_ticker_list.append({'ticker': ticker, 'stage': 'batch_flush', 'error': str(e)})
        continue

    # try:
    #     del (data, fmp_revenue_df, db_revenue_raw, db_revenue_df, rev_data,
    #          sarima_df, lstm_raw_df, prophet_raw_df, es_raw_df,
    #          all_mc, df_mc, enhanced,
    #          psr_sarima_df, psr_lstm_df, psr_prophet_df, psr_es_df,
    #          revenue_forecast_df, rf_for_db, tmp, psr_forecast_df,
    #          valuation_df, valuation_pack)
    #     gc.collect()
    # except Exception:
    #     pass

# ──────────────────────────────────────────────────────────────────────────────
# 7) 종료 요약
# ──────────────────────────────────────────────────────────────────────────────
print(f"[DONE] 성공 ticker: {total_success_tickers}")
print(f"[DONE] Valuation 업서트 rows: {total_upsert_rows}")
print(f"[DONE] Revenue forecast 업서트 rows: {total_revenue_rows}")

if error_ticker_list:
    try:
        pd.DataFrame(error_ticker_list).to_csv("valuation_error_list.csv", index=False, encoding="utf-8-sig")
        print(f"[INFO] 오류 리스트 저장: valuation_error_list.csv (총 {len(error_ticker_list)}개)")
    except Exception:
        print(f"[WARN] 오류 리스트 저장 실패 (총 {len(error_ticker_list)}개)")
else:
    print("[INFO] 오류 없이 완료")


[INFO] Project root added to sys.path: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[19:39:10] DDL: Tables ensured: us_valuation_result, us_revenue_forecast_result, us_valuation_forecast_result
[19:39:10] TICKER: 1/3 INVX


19:39:29 - cmdstanpy - INFO - Chain [1] start processing
19:39:29 - cmdstanpy - INFO - Chain [1] done processing
19:39:56 - cmdstanpy - INFO - Chain [1] start processing


[INFO] 예측 시작일: 2025-11-30 | 데이터 마지막 월: 2025-10-31


19:40:01 - cmdstanpy - INFO - Chain [1] done processing


[19:40:01] OK-VAL-PACK: INVX packed=23 batch=1
[19:40:01] TICKER: 2/3 AAPL


19:40:19 - cmdstanpy - INFO - Chain [1] start processing
19:40:19 - cmdstanpy - INFO - Chain [1] done processing
19:41:11 - cmdstanpy - INFO - Chain [1] start processing
19:41:11 - cmdstanpy - INFO - Chain [1] done processing


[INFO] 예측 시작일: 2025-11-30 | 데이터 마지막 월: 2025-10-31
[19:41:11] OK-VAL-PACK: AAPL packed=30 batch=2
[19:41:11] TICKER: 3/3 MU


19:41:36 - cmdstanpy - INFO - Chain [1] start processing
19:41:36 - cmdstanpy - INFO - Chain [1] done processing
19:42:20 - cmdstanpy - INFO - Chain [1] start processing
19:42:20 - cmdstanpy - INFO - Chain [1] done processing


[INFO] 예측 시작일: 2025-11-30 | 데이터 마지막 월: 2025-10-31
[19:42:20] OK-VAL-PACK: MU packed=30 batch=3
[19:42:20] BATCH-FLUSH: valuation=3, revenue=3, is_last=True
[19:42:20] VAL-LONG: to-insert rows=332 sample=[{'date': datetime.date(2024, 9, 30), 'ticker': 'INVX', 'indicator': 'sarima_valuation', 'value': 0.8075000000000001, 'forecate_date': datetime.date(2025, 11, 4)}, {'date': datetime.date(2024, 12, 31), 'ticker': 'INVX', 'indicator': 'sarima_valuation', 'value': 1.6099999999999999, 'forecate_date': datetime.date(2025, 11, 4)}, {'date': datetime.date(2025, 3, 31), 'ticker': 'INVX', 'indicator': 'sarima_valuation', 'value': 2.1888, 'forecate_date': datetime.date(2025, 11, 4)}]
[19:42:20] VAL-UPSERT-EXC: IntegrityError: (pymysql.err.IntegrityError) (1062, "Duplicate entry 'INVX-2025-11-04-sarima_valuation' for key 'uq_tk_fc_ind'")
[SQL: 
                                INSERT INTO us_valuation_forecast_result
                                    (`date`, `ticker`, `indicator`, `value`, `fore

In [43]:
batch_revenue_results

[]

In [45]:
revenue_forecast_df.ffill(limit=2).tail(24)

,date_month_end,revenue_billions_sarima_noexog,revenue_billions_lstm_forecast,revenue_billions_prophet_forecast,revenue_billions_esq_forecast,ticker
72,2024-06-30,0.120000,0.120000,0.120000,0.120000,INVX
73,2024-09-30,0.150000,0.150000,0.150000,0.150000,INVX
74,2024-12-31,0.250000,0.250000,0.250000,0.250000,INVX
75,2025-03-31,0.240000,0.240000,0.240000,0.240000,INVX
76,2025-06-30,0.220000,0.220000,0.220000,0.220000,INVX
77,2025-09-30,0.240000,0.240000,0.240000,0.240000,INVX
78,2025-10-31,0.240000,0.129742,0.240000,0.240000,INVX
79,2025-11-30,0.240000,0.129742,0.240000,0.240000,INVX
80,2025-12-31,0.245565,0.129742,0.136148,0.242135,INVX
81,2026-01-31,0.245565,0.139420,0.136148,0.242135,INVX


In [50]:
val_long_df

,ticker,category,model,start_month_end,start_value,end_value,growth,created_at
0,INVX,valuation,sarima,2024-09-30,None,0.807500,None,2025-11-04 09:06:37.331850
1,INVX,valuation,lstm,2024-09-30,None,0.807500,None,2025-11-04 09:06:37.331850
2,INVX,valuation,prophet,2024-09-30,None,0.807500,None,2025-11-04 09:06:37.331850
3,INVX,valuation,es,2024-09-30,None,0.807500,None,2025-11-04 09:06:37.331850
4,INVX,valuation,sarima,2024-12-31,None,1.610000,None,2025-11-04 09:06:37.331850
...,...,...,...,...,...,...,...,...
87,INVX,valuation,es,2027-02-28,None,9.540061,None,2025-11-04 09:06:37.331850
88,INVX,valuation,sarima,2027-03-31,None,4.649341,None,2025-11-04 09:06:37.331850
89,INVX,valuation,lstm,2027-03-31,None,5.951840,None,2025-11-04 09:06:37.331850
90,INVX,valuation,prophet,2027-03-31,None,1.691845,None,2025-11-04 09:06:37.331850


In [51]:
valuation_df

,date_month_end,ticker,revenue_billions_sarima_noexog_ttm,revenue_billions_lstm_forecast_ttm,revenue_billions_prophet_forecast_ttm,revenue_billions_esq_forecast_ttm,revenue_billions_avg_of_4_ttm,PSR_ttm_sarima_forecast,PSR_ttm_lstm_forecast,PSR_prophet_forecast_noexog,PSR_es_forecast,sarima_valuation,lstm_valuation,prophet_valuation,es_valuation
0,2024-09-30,INVX,0.510000,0.510000,0.510000,0.510000,0.510000,1.583333,1.583333,1.583333,1.583333,0.807500,0.807500,0.807500,0.807500
1,2024-12-31,INVX,0.630000,0.630000,0.630000,0.630000,0.630000,2.555556,2.555556,2.555556,2.555556,1.610000,1.610000,1.610000,1.610000
2,2025-03-31,INVX,0.760000,0.760000,0.760000,0.760000,0.760000,2.880000,2.880000,2.880000,2.880000,2.188800,2.188800,2.188800,2.188800
3,2025-06-30,INVX,0.860000,0.860000,0.860000,0.860000,0.860000,4.625000,4.625000,4.625000,4.625000,3.977500,3.977500,3.977500,3.977500
4,2025-09-30,INVX,0.950000,0.950000,0.950000,0.950000,0.950000,5.909091,5.909091,5.909091,5.909091,5.613636,5.613636,5.613636,5.613636
5,2025-10-31,INVX,0.940000,0.831585,0.940000,0.940000,0.912896,6.409091,6.409091,6.409091,6.409091,6.024545,5.329706,6.024545,6.024545
6,2025-11-30,INVX,0.940000,0.723171,0.940000,0.940000,0.885793,5.089160,5.148124,7.102424,5.438784,4.783810,3.722973,6.676278,5.112457
7,2025-12-31,INVX,0.965565,0.634756,0.856148,0.962135,0.854651,5.201637,5.751286,9.464021,5.706990,5.022518,3.650664,8.102602,5.490893
8,2026-01-31,INVX,0.971130,0.537443,0.752296,0.964269,0.806285,5.933390,6.368739,12.793458,5.975196,5.762092,3.422837,9.624465,5.761698
9,2026-02-28,INVX,0.976695,0.548545,0.648444,0.966404,0.785022,4.777855,7.027459,3.968426,6.243402,4.666506,3.854881,2.573301,6.033648
